# 02. Type hints, generics and Protocols

**Goal.** Use annotations as a contract between code, readers and tools. Know what Python does with them at run time.

**You will be able to:**
1. Explain why annotations are metadata and who reads them.
2. Write generic classes and functions with PEP 695 syntax (`class Stack[T]`, `def f[T: Bound]`, `type Alias[T] = ...`).
3. Define structural contracts with `Protocol` and check them statically and at run time.
4. Run a static checker and read its report.

This notebook targets Python 3.13. PEP 695 syntax is a syntax error on 3.11 and older.

## 1. Annotations are metadata

### How it works under the hood
- The compiler stores annotations in `func.__annotations__` and does nothing else. The interpreter never checks them.
- Tools read them. Static checkers (mypy, pyright) read them before run time. Libraries such as FastAPI and pydantic read them at run time through `inspect.signature` and `typing.get_type_hints`.
- `from __future__ import annotations` stores annotations as strings, so forward references need no quotes. `get_type_hints` resolves them on demand.

### Production use
- FastAPI turns `def get_user(user_id: int)` into path parsing, validation and OpenAPI docs.
- CI runs a checker so a wrong call fails a pull request, not a customer.

### Trade-offs

| Approach | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Hints only, static checker | A dict per function | Zero at run time | Documents intent | Catches bugs before release | Wrong at run time if unchecked |
| Run-time validation (pydantic, decorators) | Validator objects | Cost per call | Clear errors at the boundary | Protects untrusted input | Slower, duplicates static checks |
| No hints | None | Zero | Reader must guess | Quick scripts | Refactors break silently |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import inspect
import sys
import subprocess
import tempfile
import textwrap
import typing
from pathlib import Path


def charge(amount: float, currency: str = "GBP") -> str:
    return f"{amount:.2f} {currency}"


print("stored hints   :", charge.__annotations__)
print("wrong type still runs:", charge(12, 5))


def enforce(func):
    """Run-time check built from the same hints. Real libraries do this with more care."""
    sig = inspect.signature(func)
    hints = typing.get_type_hints(func)

    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        for name, value in bound.arguments.items():
            expected = hints.get(name)
            if expected in (int, float, str) and not isinstance(value, expected):
                raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")
        return func(*args, **kwargs)

    return wrapper


checked = enforce(charge)
print(checked(12.5))
try:
    checked("ten")
except TypeError as exc:
    print("run-time check:", exc)

source = textwrap.dedent('''
    def charge(amount: float, currency: str = "GBP") -> str:
        return f"{amount:.2f} {currency}"

    charge("ten", 5)
''')
with tempfile.TemporaryDirectory() as tmp:
    target = Path(tmp) / "demo.py"
    target.write_text(source)
    result = subprocess.run(
        [sys.executable, "-m", "mypy", "--cache-dir", str(Path(tmp) / ".cache"), "--no-error-summary", "--no-color-output", str(target)],
        capture_output=True, text=True,
    )
print("static check exit code:", result.returncode)
print(result.stdout.replace(tmp, "<tmp>").strip())

stored hints   : {'amount': <class 'float'>, 'currency': <class 'str'>, 'return': <class 'str'>}
wrong type still runs: 12.00 5
12.50 GBP
run-time check: amount must be float, got str


static check exit code: 1
<tmp>\demo.py:5: error: Argument 1 to "charge" has incompatible type "str"; expected "float"  [arg-type]
<tmp>\demo.py:5: error: Argument 2 to "charge" has incompatible type "int"; expected "str"  [arg-type]


## 2. Generics with PEP 695 syntax

### How it works under the hood
- `class TypedStack[T]` declares a type parameter in the class header. Python 3.12 added this syntax and removed the need for `TypeVar("T")` and `Generic[T]`.
- The compiler creates a `typing.TypeVar` in an annotation scope. You can read it at `TypedStack.__type_params__`.
- Variance (covariant, contravariant, invariant) is inferred from how `T` is used. Older code had to declare it by hand.
- A bound (`T: Comparable`) limits the type to subtypes of that bound. Constraints (`S: (str, bytes)`) limit it to one listed type per call.
- `type Pair[K, V] = tuple[K, V]` creates a lazily evaluated `TypeAliasType`. Its right-hand side stays unevaluated until `__value__` is read.
- Erasure: `TypedStack[int]` is only an alias object from `typing`. The running object has no element type, so the interpreter never rejects a `str`.

### Production use
- Typed repositories and result wrappers (`Repository[User]`, `Result[T, E]`) let checkers verify service layers.
- Generic helpers such as `first_or`, `chunked` and cache decorators keep their input type in the output type.

### Trade-offs

| Feature | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Generic class | One alias object per subscription | Negligible, erased at run time | Clear element types | Reusable and checked | Syntax needs 3.12 or newer |
| Bound (`T: Comparable`) | None | None | States the requirement | Rejects bad calls early | Needs a Protocol for duck-typed bounds |
| Constraints (`S: (str, bytes)`) | None | None | Explicit list | Exact per-call typing | Verbose for many types |
| Type alias statement | Small lazy object | Evaluated on demand | Names complex types | Forward references work | Not a real class at run time |

In [3]:
from understanding_python.typing_tools import Pair, TypedStack, concat, first_or, max_of

print("type params of TypedStack :", TypedStack.__type_params__)
(t_param,) = TypedStack.__type_params__
print("variance inferred         :", t_param.__infer_variance__)
print("bound of max_of's T       :", max_of.__type_params__[0].__bound__.__name__)
print("constraints of concat's S :", concat.__type_params__[0].__constraints__)
print("alias Pair                :", Pair, "->", Pair.__value__)
print("Pair[str, int]            :", Pair[str, int], "| args:", typing.get_args(Pair[str, int]))

ints = TypedStack[int]()
print("subscripted object        :", type(TypedStack[int]).__name__)
ints.push(1)
ints.push("not an int")
print("runtime accepted a str    :", list(ints))

print(max_of([3, 9, 4]), max_of(["pear", "apple"]), first_or([], "n/a"), concat(b"ab", b"cd"))
try:
    max_of([])
except ValueError as exc:
    print("empty input:", exc)

source = textwrap.dedent('''
    from understanding_python.typing_tools import TypedStack, concat

    stack = TypedStack[int]()
    stack.push("not an int")
    concat("a", b"b")
    reveal_type(stack.pop())
''')
with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / "demo.py").write_text(source)
    result = subprocess.run(
        [sys.executable, "-m", "mypy", "--cache-dir", str(Path(tmp) / ".cache"), "--no-error-summary", "--no-color-output", "demo.py"],
        capture_output=True, text=True, cwd=tmp, env={**__import__("os").environ, "MYPYPATH": str(ROOT / "src")},
    )
print(result.stdout.strip())

type params of TypedStack : (T,)
variance inferred         : True
bound of max_of's T       : Comparable
constraints of concat's S : (<class 'str'>, <class 'bytes'>)
alias Pair                : Pair -> tuple[K, V]
Pair[str, int]            : Pair[str, int] | args: (<class 'str'>, <class 'int'>)
subscripted object        : _GenericAlias
runtime accepted a str    : ['not an int', 1]
9 pear n/a b'abcd'
empty input: max_of() of empty iterable


demo.py:5: error: Argument 1 to "push" of "TypedStack" has incompatible type "str"; expected "int"  [arg-type]
demo.py:6: error: Value of type variable "S" of "concat" cannot be "Sequence[object]"  [type-var]
demo.py:7: note: Revealed type is "int"


## 3. Protocols and structural typing

### How it works under the hood
- A nominal type requires inheritance: `class Dog(Animal)`. A `Protocol` is structural: any class with the right members matches, with no base class.
- Static checkers compare method names and signatures. That is duck typing made checkable.
- `@runtime_checkable` lets `isinstance(x, Closeable)` work. It only tests that the attributes exist. It does not check signatures or types.
- Without `@runtime_checkable`, `isinstance` against a Protocol raises `TypeError`.
- `Protocol` classes carry `_is_protocol = True` and a generated `__subclasshook__`. They are never instantiated.

### Production use
- Dependency inversion: a service accepts a `Repository` Protocol. Production passes a SQL repository. Tests pass `InMemoryRepository`. Neither imports the other.
- Libraries describe plugin points (`SupportsRead`, `Comparable`) without forcing users to inherit.

### Trade-offs

| Contract style | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Protocol (structural) | Class object only | `isinstance` is slower than nominal | Contract lives in one place | No inheritance coupling | Run-time check is shallow |
| ABC (nominal) | Registry entries | `isinstance` is fast, cached | Explicit "is-a" | Enforces abstract methods at instantiation | Forces inheritance, harder for third-party types |
| Plain duck typing | None | Zero | Contract is implicit | Maximum flexibility | Errors surface only on the failing call |

In [4]:
import timeit
from abc import ABC, abstractmethod

from understanding_python.typing_tools import Closeable, InMemoryRepository, cache_user, close_all


class Connection:
    def __init__(self, name):
        self.name, self.closed = name, False

    def close(self):
        self.closed = True


class BrokenFile:
    close = "not callable"


conn = Connection("db-primary")
print("Connection matches Closeable :", isinstance(conn, Closeable))
print("int matches                  :", isinstance(3, Closeable))
print("shallow check passes a bad one:", isinstance(BrokenFile(), Closeable))
print("skipped by close_all          :", close_all([conn, 3, "log.txt"]), "| closed:", conn.closed)


class Plain(typing.Protocol):
    def ping(self) -> None: ...


try:
    isinstance(conn, Plain)
except TypeError as exc:
    print("not runtime_checkable:", exc)

repo = InMemoryRepository[str, dict]()
print(cache_user(repo, "u-42", {"name": "Ada", "plan": "pro"}))
print(cache_user(repo, "u-42", {"name": "Replaced", "plan": "free"}))


class Resource(ABC):
    @abstractmethod
    def close(self): ...


class FileResource(Resource):
    def close(self):
        pass


proto_t = timeit.timeit(lambda: isinstance(conn, Closeable), number=50_000)
file_res = FileResource()
abc_t = timeit.timeit(lambda: isinstance(file_res, Resource), number=50_000)
plain_t = timeit.timeit(lambda: isinstance(conn, Connection), number=50_000)
print(f"isinstance cost per call: Protocol {proto_t * 20:.2f} us | ABC {abc_t * 20:.2f} us | plain class {plain_t * 20:.2f} us")

Connection matches Closeable : True
int matches                  : False
shallow check passes a bad one: True
skipped by close_all          : ['int', 'str'] | closed: True
not runtime_checkable: Instance and class checks can only be used with @runtime_checkable protocols
{'name': 'Ada', 'plan': 'pro'}
{'name': 'Ada', 'plan': 'pro'}
isinstance cost per call: Protocol 0.38 us | ABC 0.24 us | plain class 0.05 us
